# **Video Akışına Otomatik Yeniden Bağlanma**

#### **Bu derste bir video akışına otomatik yeniden bağlanmayı etkinleştirmek için bir sınıfı nasıl kullanacağımızı öğreneceğiz**

In [ ]:
import cv2
import requests  
import time  

class VideoCapture:
    def __init__(self, cam_address, cam_force_address=None, blocking=False):
        """
        cam_address: ip address of the camera feed
        cam_force_address: ip address to disconnect other clients (forcefully take over)
        blocking: if true read() and connect_camera() methods blocked until ip camera is reconnected
        """
        self.cam_address = cam_address
        self.cam_force_address = cam_force_address
        self.blocking = blocking
        self.capture = None
        
        # NOT: Baskıyı azaltmak için artırılabilir
        self.RECONNECTION_PERIOD = 0.5
        # Bağlanma yöntemini çağırır
        self.connect_camera()

    def connect_camera(self):
        print("Connecting...")
        while True:
            try:
                if self.cam_force_address is not None:
                    requests.get(self.cam_force_address)

                self.capture = cv2.VideoCapture(self.cam_address)

                if not self.capture.isOpened():
                    time.sleep(self.RECONNECTION_PERIOD)
                    raise Exception("Could not connect to a camera: {0}".format(self.cam_address))

                print("Connected to a camera: {}".format(self.cam_address))

                break
            except Exception as e:
                print(e)

                if self.blocking is False:
                    break

                time.sleep(self.RECONNECTION_PERIOD)

    def getStream(self):
        """
        Reads frame and if frame is not received tries to reconnect the camera

        :return: ret - bool witch specifies if frame was read successfully
                 frame - opencv image from the camera
        """

        ret, frame = self.capture.read()

        # Besleme kesilirse yeniden bağlanmaya çalışırız
        if ret is False:
            self.connect_camera()

        return ret, frame

In [ ]:
cap = VideoCapture("rtsp://wowzaec2demo.streamlock.net/vod/mp4:BigBuckBunny_115k.mov")

while(1):
    ret, frame = cap.getStream()
    
    # Bunun, siz programı çıkmaya zorlayana kadar döngüyü çalışır durumda tutacağını unutmayın
    try:
        cv2.imshow('RTSP Stream', frame)
    except:
        print("Feed has gone down...")
        
    if cv2.waitKey(1) == 13: #13 is the Enter Key
        print("Exited...")
        break
        
# Kamerayı serbest bırakın ve pencereleri kapatın
cv2.destroyAllWindows()   

Connecting...
Connected to a camera: rtsp://wowzaec2demo.streamlock.net/vod/mp4:BigBuckBunny_115k.mov
Exited...
